# Дополнительные задания 3, 7, 16 и 24 к лабораторной работе № 1

**Дисциплина:** «Анализ данных»  
**Обучающийся:** Жойд Станислав Вадимович, группа 4417  
**Вариант:** 8

В работе рассчитываются показатели дохода и суммы кредита для разных групп заявок. Затем сравниваются процентные ставки и отбираются заявки по условиям задания 24.

## 1 Подготовка данных

Загружается файл `credit_risk_clean.csv`, полученный в основной работе. В нём должно быть 571 строка и 12 столбцов. Код проверяет размер таблицы, отсутствие пустых значений и повторяющихся строк. Столбцы с названиями категорий получают тип `category`.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', '{:.2f}'.format)

dataset_path = next((path for path in [
    Path('utility/credit_risk_clean.csv'),
    Path('credit_risk_clean.csv')
] if path.is_file()), None)

if dataset_path is None:
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(
            'Поместите credit_risk_clean.csv рядом с ноутбуком или в папку utility.'
        )
    files.upload()
    dataset_path = Path('credit_risk_clean.csv')

df = pd.read_csv(dataset_path)
for column in ['home', 'intent', 'default', 'status']:
    df[column] = df[column].astype('category')
df[['age', 'emp_length']] = df[['age', 'emp_length']].astype('int64')

assert df.shape == (571, 12)
assert not df.isna().any().any()
assert not df.duplicated().any()
print(f'Загружено {df.shape[0]} строк и {df.shape[1]} столбцов.')
display(df.head())

Загружено 571 строк и 12 столбцов.


,id,age,income,home,emp_length,intent,amount,rate,status,percent_income,default,cred_length
0,1,21,9600,OWN,5,EDUCATION,1000,11.14,0,0.10,N,2
1,2,25,9600,MORTGAGE,1,MEDICAL,5500,12.87,1,0.57,N,3
2,3,23,65500,RENT,4,MEDICAL,35000,15.23,1,0.53,N,2
3,4,24,54400,RENT,8,MEDICAL,35000,14.27,1,0.55,Y,4
4,5,21,9900,OWN,2,VENTURE,2500,7.14,1,0.25,N,2


## 2 Деление на группы

Для выполнения заданий доход, возраст и сумму кредита нужно разделить на три группы. `pd.qcut(..., q=3)` выбирает две границы по данным и присваивает каждой строке название группы. Например, доход получает одну из меток: «низкий», «средний» или «высокий».

Метод старается распределить строки примерно поровну. Но одинаковые значения попадают в одну группу, поэтому размеры групп могут отличаться. Особенно это видно для возраста: в таблице есть только значения от 21 до 26 лет.

`retbins=True` возвращает выбранные границы. Они сохраняются в переменные `income_bins`, `age_bins` и `amount_bins`, чтобы их можно было вывести и проверить.

In [2]:
income_cat = 'Категория дохода (Income)'
age_cat = 'Категория возраста'
amount_cat = 'Категория суммы кредита (Amount)'

df[income_cat], income_bins = pd.qcut(
    df['income'], q=3,
    labels=['низкий', 'средний', 'высокий'],
    retbins=True
)
df[age_cat], age_bins = pd.qcut(
    df['age'], q=3,
    labels=['молодой', 'средний', 'зрелый'],
    retbins=True
)
df[amount_cat], amount_bins = pd.qcut(
    df['amount'], q=3,
    labels=['маленькая', 'средняя', 'крупная'],
    retbins=True
)

boundaries = pd.DataFrame({
    'Признак': ['Доход', 'Возраст', 'Сумма кредита'],
    'Минимум': [income_bins[0], age_bins[0], amount_bins[0]],
    'Граница 1': [income_bins[1], age_bins[1], amount_bins[1]],
    'Граница 2': [income_bins[2], age_bins[2], amount_bins[2]],
    'Максимум': [income_bins[3], age_bins[3], amount_bins[3]],
})
display(boundaries)
for column in [income_cat, age_cat, amount_cat]:
    print(column)
    display(df[column].value_counts(sort=False).rename('Количество').to_frame())

,Признак,Минимум,Граница 1,Граница 2,Максимум
0,Доход,9600.00,54000.00,95000.00,500000.00
1,Возраст,21.00,23.00,25.00,26.00
2,Сумма кредита,1000.00,20000.00,24750.00,35000.00


Категория дохода (Income)


,Количество
Категория дохода (Income),
низкий,195
средний,187
высокий,189


Категория возраста


,Количество
Категория возраста,
молодой,247
средний,238
зрелый,86


Категория суммы кредита (Amount)


,Количество
Категория суммы кредита (Amount),
маленькая,235
средняя,145
крупная,191


Границы групп получились такими:

- Доход: «низкий» — от 9 600 до 54 000; «средний» — больше 54 000, но не больше 95 000; «высокий» — больше 95 000, до 500 000.
- Возраст: «молодой» — 21–23 года; «средний» — 24–25 лет; «зрелый» — 26 лет. Это названия групп для данного задания: «зрелый» здесь означает только возраст 26 лет.
- Сумма кредита: «маленькая» — от 1 000 до 20 000; «средняя» — больше 20 000, но не больше 24 750; «крупная» — больше 24 750, до 35 000.

Значение на границе относится к группе слева. Например, доход 54 000 считается низким, а 95 000 — средним.

## 3 Выполнение заданий

### Задание 3

**Что нужно сделать.** Для каждой группы дохода найти средний, минимальный, максимальный и медианный доход.

**Как работает код.** `groupby(income_cat)` собирает строки с одинаковой группой дохода. `['income']` выбирает столбец дохода. `agg(['mean', 'min', 'max', 'median'])` рассчитывает четыре показателя:

- `mean` — сумма доходов, делённая на число строк;
- `min` — самый маленький доход;
- `max` — самый большой доход;
- `median` — значение в середине отсортированного списка; если значений чётное число, берётся среднее двух центральных.

`round(2)` округляет результат до двух знаков после запятой. Вторая таблица показывает число строк в каждой группе.

In [3]:
result_03 = df.groupby(income_cat, observed=True)['income'].agg(
    ['mean', 'min', 'max', 'median']
).round(2)
counts_03 = df[income_cat].value_counts(sort=False).rename('count').to_frame()
display(result_03)
display(counts_03)

,mean,min,max,median
Категория дохода (Income),,,,
низкий,29533.77,9600,54000,20000.00
средний,72340.64,54400,95000,70550.00
высокий,173187.62,96000,500000,170000.00


,count
Категория дохода (Income),
низкий,195
средний,187
высокий,189


**Что получилось.**

В группе низкого дохода 195 заявок: средний доход — 29 533,77, медиана — 20 000. В группе среднего дохода 187 заявок: среднее — 72 340,64, медиана — 70 550. В группе высокого дохода 189 заявок: среднее — 173 187,62, медиана — 170 000.

В первой группе среднее заметно выше медианы. Это значит, что более высокие доходы внутри этой группы подняли среднее. В двух остальных группах разница между средним и медианой меньше. Число заявок во всех трёх группах близко.

### Задание 7

**Что нужно сделать.** Для каждой возрастной группы найти среднюю, минимальную, максимальную и медианную сумму кредита.

**Как работает код.** `groupby(age_cat)` собирает заявки по возрастным группам. `['amount']` выбирает сумму кредита. `agg` рассчитывает те же четыре показателя, что в задании 3, а `round(2)` округляет их.

В отдельной таблице подсчитывается число заявок в каждой группе: 247 для возраста 21–23 года, 238 для 24–25 лет и 86 для 26 лет.

In [4]:
result_07 = df.groupby(age_cat, observed=True)['amount'].agg(
    ['mean', 'min', 'max', 'median']
).round(2)
counts_07 = df[age_cat].value_counts(sort=False).rename('count').to_frame()
display(result_07)
display(counts_07)

,mean,min,max,median
Категория возраста,,,,
молодой,17777.23,1000,35000,21000.00
средний,20107.98,1200,35000,22000.00
зрелый,20562.79,1500,35000,22250.00


,count
Категория возраста,
молодой,247
средний,238
зрелый,86


**Что получилось.**

У заявителей 21–23 лет средняя сумма кредита — 17 777,23, медиана — 21 000. Для возраста 24–25 лет среднее — 20 107,98, медиана — 22 000. Для 26 лет среднее — 20 562,79, медиана — 22 250. Самая большая сумма во всех трёх группах — 35 000.

В этой таблице более старшие группы в среднем берут немного большие кредиты. Но по одному сравнению нельзя сказать, что причина именно в возрасте: на сумму могут влиять и другие данные о заявителе.

### Задание 16

**Что нужно сделать.** Узнать среднюю процентную ставку для каждой пары групп: суммы кредита и дохода.

**Как работает код.** `pivot_table` строит таблицу, где:

- строки (`index=amount_cat`) — группы суммы кредита;
- столбцы (`columns=income_cat`) — группы дохода;
- значения (`values='rate'`, `aggfunc='mean'`) — средние ставки.

Например, ячейка на пересечении строки «маленькая» и столбца «средний» показывает среднюю ставку по небольшим кредитам у заявителей со средним доходом. Вторая таблица, с `aggfunc='count'`, показывает, по скольким заявкам рассчитано каждое среднее.

In [5]:
result_16 = df.pivot_table(
    index=amount_cat, columns=income_cat, values='rate',
    aggfunc='mean', observed=True
).round(2)
counts_16 = df.pivot_table(
    index=amount_cat, columns=income_cat, values='rate',
    aggfunc='count', observed=True
)
display(result_16)
display(counts_16)

Категория дохода (Income),низкий,средний,высокий
Категория суммы кредита (Amount),,,
маленькая,11.56,10.76,11.86
средняя,12.33,12.69,13.28
крупная,13.27,13.16,12.31


Категория дохода (Income),низкий,средний,высокий
Категория суммы кредита (Amount),,,
маленькая,151,8,76
средняя,21,80,44
крупная,23,99,69


**Что получилось.**

Самая низкая средняя ставка — 10,76 — у маленьких кредитов в группе среднего дохода. Самая высокая — 13,28 — у средних кредитов в группе высокого дохода.

Для крупных кредитов ставка равна 13,27 при низком доходе, 13,16 при среднем и 12,31 при высоком. В этой строке ставка снижается с ростом дохода. Но в других строках порядок отличается, поэтому общего правила «чем выше доход, тем ниже ставка» таблица не показывает.

### Задание 24

**Что нужно сделать.** Отобрать заявки по ставке, цели кредита и типу жилья. Для оставшихся заявок найти медианную, среднюю и максимальную сумму кредита.

**Как работает код.** Отбор выполняется по шагам:

1. `quantile([0.25, 0.75])` находит две границы ставки. Если отсортировать ставки, первая граница будет примерно на четверти списка, а вторая — на трёх четвертях. Получились 10,36 и 14,74.
2. `between(rate_low, rate_high)` оставляет заявки со ставкой от 10,36 до 14,74 включительно.
3. Среди оставшихся заявок рассчитывается средний доход для каждой цели кредита. `nlargest(2)` выбирает две цели с самым высоким средним доходом.
4. Для этих двух целей `value_counts()` считает заявки по типам жилья. Типы, которых нет в оставшихся строках, исключаются. `tail(1)` выбирает самый редкий из оставшихся типов.
5. `pivot_table` рассчитывает медианную, среднюю и максимальную сумму кредита. Строки таблицы — выбранные цели, столбцы — выбранный тип жилья.

In [6]:
rate_low, rate_high = df['rate'].quantile([0.25, 0.75])
task_24_rate = df[df['rate'].between(rate_low, rate_high)].copy()
top_2_intents_24 = task_24_rate.groupby(
    'intent', observed=True
)['income'].mean().nlargest(2).index
task_24_by_intent = task_24_rate[task_24_rate['intent'].isin(top_2_intents_24)]
home_counts_24 = task_24_by_intent['home'].value_counts()
home_counts_24 = home_counts_24[home_counts_24 > 0]
least_home_24 = home_counts_24.tail(1).index[0]
task_24_filtered = task_24_by_intent[task_24_by_intent['home'].eq(least_home_24)]
result_24 = task_24_filtered.pivot_table(
    index='intent', columns='home', values='amount',
    aggfunc=['median', 'mean', 'max'], observed=True
).round(2)
print(f'Диапазон rate: [{rate_low:.2f}; {rate_high:.2f}]')
print(f'В диапазоне: {len(task_24_rate)} строк')
print(f'Топ-2 целей: {list(top_2_intents_24)}')
print(f'Наименее частый представленный home: {least_home_24}')
display(result_24)

Диапазон rate: [10.36; 14.74]
В диапазоне: 287 строк
Топ-2 целей: ['HOMEIMPROVEMENT', 'MEDICAL']
Наименее частый представленный home: OWN


,median,mean,max
home,OWN,OWN,OWN
intent,,,
HOMEIMPROVEMENT,4037.50,8595.83,28000
MEDICAL,4000.00,8371.43,20000


**Что получилось.**

После отбора по ставке осталось 287 заявок. Самый высокий средний доход у заявителей с целями `HOMEIMPROVEMENT` (улучшение жилья) и `MEDICAL` (медицинские расходы). Среди этих двух целей 50 заявок относятся к `RENT` (аренда), 18 — к `MORTGAGE` (ипотека) и 13 — к `OWN` (собственное жильё). Поэтому выбран `OWN` — самый редкий тип жилья в оставшихся заявках.

Для улучшения жилья осталось 6 заявок: медианная сумма кредита — 4 037,50, средняя — 8 595,83, максимальная — 28 000. Для медицинских расходов осталось 7 заявок: медиана — 4 000, среднее — 8 371,43, максимум — 20 000.

Средние почти вдвое выше медиан, потому что отдельные большие кредиты увеличивают среднее. Заявок мало — всего 6 и 7, поэтому эти значения описывают только оставшиеся строки.

## 4 Итоговый контроль

Код проверяет, что рассчитаны все четыре задания, границы групп совпадают с полученными значениями, после отбора по ставке осталось 287 заявок и итоговые таблицы не пустые.

In [7]:
assert all(abs(a - b) < 1e-6 for a, b in zip(
    income_bins, [9600.0, 54000.0, 95000.0, 500000.0]
))
assert all(abs(a - b) < 1e-6 for a, b in zip(
    age_bins, [21.0, 23.0, 25.0, 26.0]
))
assert all(abs(a - b) < 1e-6 for a, b in zip(
    amount_bins, [1000.0, 20000.0, 24750.0, 35000.0]
))
assert len(task_24_rate) == 287
assert all(name in globals() for name in ['result_03', 'result_07', 'result_16', 'result_24'])
assert all(name in globals() for name in ['counts_03', 'counts_07', 'counts_16'])
assert not result_03.empty and not result_07.empty
assert not result_16.empty and not result_24.empty

control = pd.DataFrame({
    'Проверка': [
        'Число исходных строк',
        'Строк в диапазоне ставки задания 24',
        'Строк в итоговой выборке задания 24',
        'Число выполненных заданий',
        'Ошибки выполнения'
    ],
    'Результат': [len(df), len(task_24_rate), len(task_24_filtered), 4, 0]
})
display(control)
print('Задания 3, 7, 16 и 24 выполнены, контрольные проверки пройдены.')

,Проверка,Результат
0,Число исходных строк,571
1,Строк в диапазоне ставки задания 24,287
2,Строк в итоговой выборке задания 24,13
3,Число выполненных заданий,4
4,Ошибки выполнения,0


Задания 3, 7, 16 и 24 выполнены, контрольные проверки пройдены.


## ВЫВОД

Выполнены задания 3, 7, 16 и 24 по таблице из 571 кредитной заявки. Доход, возраст и сумма кредита разделены на три группы. Границы рассчитаны по значениям в таблице с помощью `pd.qcut`.

В заданиях 3 и 7 для каждой группы найдены среднее, минимум, максимум и медиана. В задании 16 рассчитана средняя ставка для каждой пары групп суммы кредита и дохода. Самая низкая средняя ставка — 10,76, самая высокая — 13,28.

В задании 24 после отбора по ставке осталось 287 заявок. Затем выбраны две цели с самым высоким средним доходом и самый редкий тип жилья — `OWN`. В итоге осталось 6 заявок на улучшение жилья и 7 на медицинские расходы. Для них рассчитаны медианная, средняя и максимальная суммы кредита.